## Multi-Tooling with Foundry Agents

### Installing Utilities and Libraries

In [ ]:
%pip install azure-ai-projects==2.7.0 openai==1.109.1 python-dotenv azure-identity jsonref

![multi-tool-agent](./Images/multi-tool-lab-flow.png)

### Setting up the Environment Variables

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("CHAT_MODEL_DEPLOYMENT_NAME")

### Setting up the Foundry Project Client

In [ ]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

### Creating the OpenAI Client

In [ ]:
openai_client = project_client.get_openai_client()

### Initializing the OpenAPI Tool

In [ ]:
import jsonref

with open("./weather_openapi.json", "r") as f:
        openapi_weather = jsonref.loads(f.read())

# Initialize agent OpenApi tool using the read in OpenAPI spec
weather_tool = {
        "type": "openapi",
        "openapi":{
            "name": "weather",
            "spec": openapi_weather,
            "auth": {
                "type": "anonymous"
            },
        }
}

### Creating the Code Interpreter Tool

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition, CodeInterpreterTool, AutoCodeInterpreterToolParam

code_interpreter_tool = CodeInterpreterTool(container = AutoCodeInterpreterToolParam())

### Creating the Multi-Tool Agent

In [ ]:
agent_name = "multi-tool-agent"

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="You are a helpful assistant that can use multiple tools to answer user queries.",
        tools=[weather_tool, code_interpreter_tool],
    )
)

print(f"Created agent: {agent.name} with ID: {agent.id}")

### Creating a Conversation Object for the Agent Chat System

In [ ]:
# create a conversation to use with the agent
conversation = openai_client.conversations.create()
print(f"Created conversation with id: {conversation.id}")

### Calling our Agent

In [ ]:
user_query = "Let me know the weather in Seattle and also find the standard deviation of the first ten natural numbers"

In [ ]:
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body = {
        "agent_reference": {
            "name": agent_name,
            "type": "agent_reference"
        }
    },
    input = user_query
)

print(f"Agent response: {response.output_text}")